# noise_reachability step 13: do the distilled adapters change success on initial states they never trained on? (Colab T4)

Step 13 of `noise_reachability/README.md`. The base, step 10's N<t>_0-2 and step 12's R<t>_0-2 on initial states 30-49 of
tasks 0, 6 and 7 that no adapter trained on, seeds 200-207, sampled noise (2848 episodes; R is not run on task 7,
whose R weights are N's).

**Inputs:** the archive on Google Drive, `MyDrive/Alienware_Research_archives/efficient_vla/` with
`noise-reachability-step-10/` and `expert-iteration-step-12/` (as `tools/archive_kaggle_output.sh` wrote them). Step 6
checks every adapter's sha256 against the archive manifests.

**Runtime: T4 GPU.** Set the session length `STOP_AT_H` in step 1 (hours, default 20; e.g. 8 for a working day), then **Run all**. Every session first re-runs 12 of step 10's Kaggle T4 episodes
(harness) and stops if fewer than 11 reproduce. Keep the tab open; about 45 h of runtime in all (estimate), so several
sessions: after a disconnect or the end of a session, **Run all** again. Finished episodes are kept on Drive and skipped.

When step 10 says every episode is done, download `step13_results.zip`.

At the end of every session, step 11 downloads `step13_results.zip` and step 12 syncs Drive and, after 90 s, releases
the runtime so it stops using compute units (set `RELEASE_RUNTIME = False` there to keep it).

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "26fe1d481941d652fb6a7319d7c0e4bb3a458d4e"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 13; nothing here but STOP_AT_H is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [0, 6, 7]
N_ACTION_STEPS = 10
STATES = {0: '30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49', 6: '30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49', 7: '30,31,32,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49'}   # 30-49 minus trained states (7/33)
PARTS = ['200-203', '204-207']                 # seeds 200-207 in two parts per policy and task
POLICIES = ['base', 'N0', 'N1', 'N2', 'R0', 'R1', 'R2']
RUN_POLICIES = {0: ['base', 'N0', 'N1', 'N2', 'R0', 'R1', 'R2'], 6: ['base', 'N0', 'N1', 'N2', 'R0', 'R1', 'R2'], 7: ['base', 'N0', 'N1', 'N2']}   # R7_s's weights are N7_s's: not run
HARNESS_STATE = {0: 8, 6: 8, 7: 33}   # every session: N<t>_0 on the Colab T4 check's step-10 episodes
HARNESS_SEEDS = "16-19"
ARCHIVE = "/content/drive/MyDrive/Alienware_Research_archives/efficient_vla"   # tools/archive_kaggle_output.sh folders
RUNS_IN = {"N": f"{ARCHIVE}/noise-reachability-step-10/output/step10/runs",
           "R": f"{ARCHIVE}/expert-iteration-step-12/output/step12/runs"}
ADAPTER_SHA256 = {'N0_0': {'adapter_config.json': '4f512e8d34de097adadb1d1e76324556a58d5a486dd98169929a772801a146fd', 'adapter_model.safetensors': 'aa163591bfc41a54da6d711908d828f7fe5b9a8d7d9f8895f1392c07e3e2d48d', 'run_config.json': 'faf912a80ab9fd77669633970d54325c0d51fbe8f6bad968e1e34d7448cd0f01'}, 'N0_1': {'adapter_config.json': '384a41facbd1070d02158a334ec5b3d4ddebcdbd9a66361755450a2a6ab09ff7', 'adapter_model.safetensors': '7b7c89e575a72cdee3084260acb3a9642f8a6a6ef4511147ecdb4da43123c57b', 'run_config.json': 'faf912a80ab9fd77669633970d54325c0d51fbe8f6bad968e1e34d7448cd0f01'}, 'N0_2': {'adapter_config.json': '4f6630bd090b14e00deab8a0a79a37a924c64a28aa23f0d58ac743372799b404', 'adapter_model.safetensors': '295237a3a12988fdfceaa5643e8c58a78cb5f1c16d7c84ddc1f5ee9b655e4e41', 'run_config.json': 'faf912a80ab9fd77669633970d54325c0d51fbe8f6bad968e1e34d7448cd0f01'}, 'N6_0': {'adapter_config.json': '6e3e0f49f27fc0fef5a0d76761a971b2c5c4feec91ea81ee8cd41de657c69cc3', 'adapter_model.safetensors': '1eb3b7d5321c1a1d7455bc491920a2f2dfcf9542b81a91751163139c16eb0498', 'run_config.json': '16af5136a897ace291ef2b255db723f3614b81d318b7df73ee16977a1e7bed8c'}, 'N6_1': {'adapter_config.json': '4cb696c73129fc0e437255d80aa7e69a902d9742ac3c51657e0f5697dac0002e', 'adapter_model.safetensors': '89b9a705b7d1ebdea8b4c180e5618f95cf4fbc24d9ce3d67c2a6bd167c931c52', 'run_config.json': '16af5136a897ace291ef2b255db723f3614b81d318b7df73ee16977a1e7bed8c'}, 'N6_2': {'adapter_config.json': 'ac688cf422ae58c00b8e992d41fbd1056e397124100b198e8f1866dba7ed1432', 'adapter_model.safetensors': '71e65d76a9b74ac50b0748ceddeb1c7526cf89e03959ed87b4c666a0ef7fdbe7', 'run_config.json': '16af5136a897ace291ef2b255db723f3614b81d318b7df73ee16977a1e7bed8c'}, 'N7_0': {'adapter_config.json': '33f536c16b513ab6673fa7106a9e38be80df456a246ccef0131f4d85aefdc91e', 'adapter_model.safetensors': '49c56ea1ea44cb6d4dd50c6362f4304531af410c355b03917c5cafa94bd00c40', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}, 'N7_1': {'adapter_config.json': '903aae691044be234e4887dd43dce0c59962576684ce91f10451380009f5b7cd', 'adapter_model.safetensors': '497fa7e300c2a3650a3091016f92f6607400a0cdd13ad56f065e4efdb1347366', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}, 'N7_2': {'adapter_config.json': 'cc5ca0f30c9e85ef175afaa668532da3c7bcd92b60fa6e164ef1212a5e57b5c0', 'adapter_model.safetensors': '41a132f12b07fd7652b0d0b8692051eb1a8a4887e578552a832122aab3db7ea3', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}, 'R0_0': {'adapter_config.json': 'c05ec2f485970f353ad97fe5989a05d430168e7609149b7b1940daf5b9b0e827', 'adapter_model.safetensors': '5eef27693909dfafe5b49335fbb1398a6f73110eeb4eb4064009bb5843ae6d0c', 'run_config.json': 'faf912a80ab9fd77669633970d54325c0d51fbe8f6bad968e1e34d7448cd0f01'}, 'R0_1': {'adapter_config.json': 'd41a94803aa215de7fa29329bc944827e83ea9f293750141bc836437e4714270', 'adapter_model.safetensors': '26351fd57db533b16b798cc3b61eb48b0bdb10ee3a98b328a94461cd0ea65774', 'run_config.json': 'faf912a80ab9fd77669633970d54325c0d51fbe8f6bad968e1e34d7448cd0f01'}, 'R0_2': {'adapter_config.json': '312d708ea0eae4b6aad0858546f53199630038619c3434927e5ee8ba8756a93a', 'adapter_model.safetensors': 'd9d605d7d32d493c82c41a00df1091ba16d62abd60889c92b17d7e6fd399eebb', 'run_config.json': 'faf912a80ab9fd77669633970d54325c0d51fbe8f6bad968e1e34d7448cd0f01'}, 'R6_0': {'adapter_config.json': 'a8d2f91f3aa49ec89d1b7bc46cd66a22406f738c10db9ae414fc6e5154b3062e', 'adapter_model.safetensors': '2fad4bb1d99d12fca8235a699f26e7f271e925b9b76b0feb41f38cefa098bfb4', 'run_config.json': '16af5136a897ace291ef2b255db723f3614b81d318b7df73ee16977a1e7bed8c'}, 'R6_1': {'adapter_config.json': 'b412ef733b3becd20f2bb3f5dd7e7708f6686326bd6043f306f7a46ea38a559c', 'adapter_model.safetensors': '9d2dc443e06ebca0e034f9bc8465b2092ed4f49351eee708758c39c89c5fb8ab', 'run_config.json': '16af5136a897ace291ef2b255db723f3614b81d318b7df73ee16977a1e7bed8c'}, 'R6_2': {'adapter_config.json': '284e6f8fe7b8a98c6a18ad39b1656aafe5fb3af8ad3ef6e45ab4b2ad83e018ca', 'adapter_model.safetensors': '4a98d39be880b2f717ef77e9a954af63032a5d90ca141fb02e7dfd0cade345f7', 'run_config.json': '16af5136a897ace291ef2b255db723f3614b81d318b7df73ee16977a1e7bed8c'}, 'R7_0': {'adapter_config.json': '94e9c4ac6d237e221e114d5dd044c413072aa1218c398a09de28996aad039460', 'adapter_model.safetensors': '49c56ea1ea44cb6d4dd50c6362f4304531af410c355b03917c5cafa94bd00c40', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}, 'R7_1': {'adapter_config.json': '5337ac193ddaafc0e7b30b601eede1b8edfcb4fab7a0731048e5897a9438a377', 'adapter_model.safetensors': '497fa7e300c2a3650a3091016f92f6607400a0cdd13ad56f065e4efdb1347366', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}, 'R7_2': {'adapter_config.json': '2f5511492a8adfbac42c40088f04d8b8180d471c4c15eec8a1d3af9be858d0d3', 'adapter_model.safetensors': '41a132f12b07fd7652b0d0b8692051eb1a8a4887e578552a832122aab3db7ea3', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}}
DRIVE_DIR = "/content/drive/MyDrive/efficient_vla_runs/step13"
OUT_ROOT = f"{DRIVE_DIR}/run"
STOP_AT_H = 20.0  #@param {type:"number"}
# STOP_AT_H: the session's length in hours, the one setting meant to be edited (also in the form on the right). At that
# time no new episodes start, running ones are stopped (re-run next session), then the zip downloads and step 12
# releases the runtime. Unfinished work continues with the next Run all.
assert 0.5 <= STOP_AT_H <= 24, f"STOP_AT_H = {STOP_AT_H}: choose 0.5-24 hours"
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 13 at {REPO_REF}: tasks {TASKS}, 59 states, seeds 200-207, policies "
      f"{POLICIES}, 2848 evaluation episodes to run plus 12 harness episodes per session, "
      f"stop after {STOP_AT_H} h, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. GPU, RAM and Drive

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus and all("T4" in g[0] for g in gpus), f"GPU {gpus}: set the runtime type to T4 GPU (step 13 runs on T4 only)"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(4, int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
print(f"{len(gpus)} GPU(s) {gpus}, driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"{avail_gb:.0f} GB RAM available -> {JOBS} evaluation(s) at a time")

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{OUT_ROOT}/logs", exist_ok=True)

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment (the package versions of the Kaggle runs; several minutes)

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. This session, its commit, and the adapters (sha256 against the archive manifests)

In [ ]:
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
earlier = first.read_text().split() if first.exists() else []
if earlier and earlier[0] != head:
    raise RuntimeError(f"step 13 started at commit {earlier[0]}, this session cloned {head}")
with open(first, "a") as f:
    f.write(head + "\n")
k = len(earlier) + 1
while any(Path(OUT_ROOT, d, f"session{k}").exists() for d in ("harness", "harness_pending", "harness_failed")):
    k += 1                                   # a session whose repo_commit.txt line did not reach Drive
SESSION = f"session{k}"
Path(OUT_ROOT, f"environment_{SESSION}.json").write_text(json.dumps(
    {"gpus": gpus, "cuda_driver": CUDA_DRIVER, "torch_build": TORCH_BUILD, "ram_available_gb": round(avail_gb, 1),
     "started": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=1) + "\n")
print("this is", SESSION)


def adapter_dir(t, policy):
    return f"{RUNS_IN[policy[0]]}/{policy[0]}{t}_{policy[1:]}/adapter"


for name, files in ADAPTER_SHA256.items():
    folder = adapter_dir(int(name[1]), name[0] + name[3:])
    for fname, want in files.items():
        p = Path(folder, fname)
        assert p.exists(), f"{p} is missing: upload the archive folder to Drive"
        got = hashlib.sha256(p.read_bytes()).hexdigest()
        assert got == want, f"{p}: sha256 {got}, expected the archive's {want}"
    trained_with = json.loads(Path(folder, "run_config.json").read_text())["policy_path"]
    assert trained_with == POLICY_PATH, f"{name} was trained with {trained_with}, this session has {POLICY_PATH}"
print(f"{len(ADAPTER_SHA256)} adapters: sha256 equal to the archive's, trained with this checkpoint path")

## 7. Helpers: jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 11, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






MAIN = f"{REPO}/self_filtered_bc/self_filtered_bc.py"
ANALYZE = f"cd {REPO} && python3 -S noise_reachability/analyze_step13.py --root {OUT_ROOT}"


def sfbc(task):
    return f"cd {REPO} && {PY} -u {MAIN} --task_id {task} --n_action_steps {N_ACTION_STEPS} --policy_path {POLICY_PATH}"


def suffix(policy):
    return "base" if policy == "base" else "lora"


def n_done(path):
    """Distinct episodes in a per-episode file (a row count would hide duplicates)."""
    return len({(r["seed"], r["init_state"]) for r in read_csv(path)})


def stop_leftovers():
    """Jobs run in their own process groups, so an interrupted cell can leave them writing to the same files."""
    sh("pkill -f 'self_filtered_bc/self_filtered_bc[.]py'; sleep 5", check=False)


def eval_out(policy, t, j):
    return f"{OUT_ROOT}/eval/{policy}/task_{t}/part{j}"


def eval_cmd(policy, t, j):
    which = "--eval_policies base" if policy == "base" else f"--eval_policies lora --lora_dir {adapter_dir(t, policy)}"
    return (f"{sfbc(t)} --stage eval {which} --eval_seeds {PARTS[j]} --eval_init_states {STATES[t]} "
            f"--eval_output_dir {eval_out(policy, t, j)} --max_episodes_per_process 10")


def part_size(t, j):
    return n_range(PARTS[j]) * len(STATES[t].split(","))


def harness_cmd(t):
    return (f"{sfbc(t)} --stage eval --eval_policies lora --lora_dir {adapter_dir(t, 'N0')} --eval_seeds {HARNESS_SEEDS} "
            f"--eval_init_states {HARNESS_STATE[t]} --eval_output_dir {OUT_ROOT}/harness_pending/{SESSION}/task_{t} "
            f"--max_episodes_per_process 10")

## 8. Harness of this session: N<t>_0 on 12 of step 10's Kaggle T4 episodes; stops the session if it fails

In [ ]:
stop_leftovers()
PENDING = f"{OUT_ROOT}/harness_pending/{SESSION}"
jobs = {f"harness_{SESSION}_task{t}": (harness_cmd(t), 0) for t in TASKS
        if n_done(f"{PENDING}/task_{t}/per_episode_lora.csv") < n_range(HARNESS_SEEDS)}
codes = run_jobs(jobs, max_parallel=JOBS)
HARNESS_OK = False
if failed(codes) or any(c is None for c in codes.values()):
    fail(f"harness jobs failed: {codes}")      # left in harness_pending/, which the analysis does not read
elif (gate_rc := sh(f"{ANALYZE} --gate harness_pending/{SESSION}", check=False).returncode) == 1:
    os.makedirs(f"{OUT_ROOT}/harness_failed", exist_ok=True)
    os.rename(PENDING, f"{OUT_ROOT}/harness_failed/{SESSION}")
    fail("the harness failed: this session does not reproduce step 10 or has other package versions; no episodes added")
elif gate_rc != 0:
    fail("the harness could not be read (left in harness_pending/): see the message above")
else:
    os.makedirs(f"{OUT_ROOT}/harness", exist_ok=True)
    os.rename(PENDING, f"{OUT_ROOT}/harness/{SESSION}")
    HARNESS_OK = True

## 9. Evaluation: 2848 episodes, one part (4 seeds x one task's states) per job, in an order that keeps policies even

In [ ]:
def eval_done(policy, t):
    return sum(n_done(f"{eval_out(policy, t, j)}/per_episode_{suffix(policy)}.csv") for j in range(len(PARTS)))


if HARNESS_OK and not FAILURE:
    stop_leftovers()
    jobs = {}
    for j in range(len(PARTS)):
        for t in TASKS:
            for policy in RUN_POLICIES[t]:
                if n_done(f"{eval_out(policy, t, j)}/per_episode_{suffix(policy)}.csv") < part_size(t, j):
                    jobs[f"eval_{policy}_task{t}_part{j}"] = (eval_cmd(policy, t, j), 0)
    total = sum(len(RUN_POLICIES[t]) * len(PARTS) for t in TASKS)
    print(f"evaluation: {len(jobs)} of {total} parts to run; {hours_left():.1f} h left", flush=True)
    codes = run_jobs(jobs, max_parallel=JOBS)
    if failed(codes):
        fail(f"evaluation failed: {failed(codes)}; see {OUT_ROOT}/logs")
done = {t: {p: eval_done(p, t) for p in RUN_POLICIES[t]} for t in TASKS}
print("episodes done:", done)
ALL_DONE = all(done[t][p] == sum(part_size(t, j) for j in range(len(PARTS))) for t in TASKS for p in RUN_POLICIES[t])

## 10. Analysis (when every episode is done)

In [ ]:
if ALL_DONE and not FAILURE:
    if sh(f"{ANALYZE} --out {OUT_ROOT}/step13_report.txt", check=False).returncode != 0:
        fail("the analysis failed: the episodes are on Drive and in the zip")
elif not FAILURE:
    print("WORK REMAINS: Run all again in a new session (finished episodes are kept on Drive)")

## 11. Download

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /content/step13_results.zip . -x '*.pyc' -x '*.tmp*'")
from google.colab import files
files.download("/content/step13_results.zip")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")

## 12. Release the runtime (stops the compute-unit charge; the results are on Drive and in the zip)

In [ ]:
RELEASE_RUNTIME = True             # False keeps the GPU assigned after the session's work
if FAILURE:
    print("THIS SESSION FAILED:", FAILURE)
drive.flush_and_unmount()          # every episode file written to Drive before the machine goes
print("Drive synced and unmounted")
if RELEASE_RUNTIME:
    print("releasing the runtime in 90 s (time for the browser to save step13_results.zip)", flush=True)
    time.sleep(90)
    from google.colab import runtime
    runtime.unassign()
if FAILURE:
    raise RuntimeError(FAILURE)